## tl;dr

El benchmark disponible es **sintético**, por lo que no demuestra qué algoritmo gana en la cartera Freddie real. En esta ejecución la regresión logística gana la selección PD, el baseline constante gana EAD y el Huber de LGD es elegido por error agregado pese a tener peor error individual. La prioridad es corregir validación temporal, madurez y criterio de selección antes de añadir familias de modelos.

## Contexto y métodos

Notebook de diagnóstico que resume `outputs/study-results.json`, generado con `python -m scripts.run_study --mode synthetic`. No reentrena challengers nuevos ni extrapola los resultados sintéticos a una cartera real.

### Supuestos clave

- Menor Brier y log-loss es mejor; mayor ROC-AUC es mejor.
- Las métricas de EAD/LGD se comparan solo dentro del mismo componente y ventana.
- La selección operativa requiere una prueba temporal con datos reales y etiquetas maduras.

## Datos

In [ ]:
from pathlib import Path
import json
import pandas as pd

root = Path.cwd().resolve()
while root != root.parent and not (root / 'outputs' / 'study-results.json').is_file():
    root = root.parent
result_path = root / 'outputs' / 'study-results.json'
if not result_path.is_file():
    raise FileNotFoundError('Ejecuta primero: python -m scripts.run_study --mode synthetic')
study = json.loads(result_path.read_text(encoding='utf-8'))
assert study['identity']['source'] == 'generated_in_memory'
print({'source': study['identity']['source'], 'rows': study['identity']['rows'], 'path': str(result_path)})

## Resultados

### 1. Comparación PD en validación 2020

In [ ]:
pd_rows = [
    {'modelo': name, 'features': 'base', **metrics}
    for name, metrics in study['pd']['validation_metrics'].items()
]
macro = study['pd']['macro_challenger']
pd_rows.append({'modelo': macro['selected_name'], 'features': 'base + macro', **macro['validation_metrics']})
pd_comparison = pd.DataFrame(pd_rows)[
    ['modelo', 'features', 'n', 'events', 'roc_auc', 'pr_auc', 'brier', 'log_loss', 'calibration_intercept', 'calibration_slope']
].sort_values(['brier', 'log_loss', 'roc_auc'], ascending=[True, True, False])
print(pd_comparison.to_string(index=False))
base = study['pd']['validation_metrics']
assert study['pd']['selected_name'] == 'logistic'
print({
    'auc_logistic_minus_hgb': base['logistic']['roc_auc'] - base['hist_gradient_boosting']['roc_auc'],
    'brier_logistic_minus_hgb': base['logistic']['brier'] - base['hist_gradient_boosting']['brier'],
    'macro_promoted': macro['promoted'],
})

### 2. Comparación EAD y LGD en sus ventanas de validación

In [ ]:
loss_rows = []
for component in ('ead', 'lgd'):
    section = study['loss_components'][component]
    for name, metrics in section['validation_metrics'].items():
        loss_rows.append({'componente': component.upper(), 'modelo': name, 'seleccionado': name == section['selected_name'], **metrics})
loss_comparison = pd.DataFrame(loss_rows)[
    ['componente', 'modelo', 'seleccionado', 'n', 'mae', 'rmse', 'wape', 'portfolio_relative_error']
].sort_values(['componente', 'portfolio_relative_error', 'mae'])
print(loss_comparison.to_string(index=False))
lgd = study['loss_components']['lgd']['validation_metrics']
assert study['loss_components']['ead']['selected_name'] == 'constant_1'
assert study['loss_components']['lgd']['selected_name'] == 'direct_huber'
assert lgd['direct_huber']['mae'] > min(lgd['hurdle']['mae'], lgd['segment_mean']['mae'])
print({'decision_grade': study['loss_components']['decision_grade']})

## Takeaways

- La logística es el champion PD defendible con la evidencia actual; HGB queda como challenger, no como sustituto probado.
- EAD no justifica complejidad en la muestra sintética: el valor constante 1 supera al HGB.
- La selección LGD prioriza el error neto de cartera y puede premiar cancelación de errores; debe añadirse un gate de MAE/RMSE y calibración por segmento.
- El resultado `decision_grade = false` y la ausencia del fichero Freddie real impiden una recomendación de producción.